In [4]:
import os
import time
import pandas as pd
from urllib.parse import urlparse, parse_qs

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager


driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

base_url = "https://www.saramin.co.kr"
search_url = (
    "https://www.saramin.co.kr/zf_user/search"
    "?searchType=search&searchword=데이터분석"
)

driver.get(search_url)
time.sleep(2)  


items = driver.find_elements(By.CSS_SELECTOR, "div.item_recruit")
print("공고 개수:", len(items))

rows = []

for item in items:
    try:
        
        try:
            company_el = item.find_element(
                By.CSS_SELECTOR,
                ".area_corp a, .area_corp strong, .corp_name a"
            )
            col_company = company_el.text.strip()
        except:
            col_company = ""

        
        try:
            title_el = item.find_element(
                By.CSS_SELECTOR,
                "div.area_job h2.job_tit a"
            )
        except:
            
            continue

        col_recruit = title_el.text.strip()
        raw_href = title_el.get_attribute("href") or ""


        col_url = raw_href  

        if raw_href:
            parsed = urlparse(raw_href)
            qs = parse_qs(parsed.query)
            rec_idx = qs.get("rec_idx", [None])[0]

            if rec_idx:
                
                col_url = (
                    f"{base_url}/zf_user/jobs/relay/view"
                    f"?view_type=search&rec_idx={rec_idx}"
                )


        try:
            cond_elements = item.find_elements(
                By.CSS_SELECTOR,
                ".job_condition span"
            )
            cond_texts = [
                c.text.strip()
                for c in cond_elements
                if c.text.strip()
            ]
            col_detail = " | ".join(cond_texts)
        except:
            col_detail = ""

        rows.append(
            {
                "Site": "saramin",
                "Col_Company": col_company,
                "Col_Recruit": col_recruit,
                "Col_detail": col_detail,
                "Col_url": col_url,
            }
        )

    except Exception as e:
        print("Error on one card:", e)
        continue

driver.quit()

df = pd.DataFrame(rows)

output_path = os.path.join("data_tmp", "data_saramin.csv")
df.to_csv(output_path, index=False, encoding="utf-8-sig")

output_path, df.head()

공고 개수: 40


('data_tmp\\data_saramin.csv',
       Site     Col_Company                  Col_Recruit  \
 0  saramin         트리노드(주)  [서울/경력 5년 이상] 데이터분석팀 데이터분석가   
 1  saramin  콘센트릭스서비스코리아(유)       [Catalyst] GA/AA 데이터분석   
 2  saramin       넛지헬스케어(주)     [캐시워크] 데이터분석 담당 채용전환형 인턴   
 3  saramin          (주)미리디                [미리디] 데이터 분석가   
 4  saramin       넛지헬스케어(주)        [캐시워크] 데이터분석가 - 석사/박사   
 
                       Col_detail  \
 0    서울 강남구 | 경력5년↑ | 학력무관 | 정규직   
 1    서울 강남구 | 경력1년↑ | 학력무관 | 정규직   
 2        서울 강남구 | 신입 | 대졸↑ | 인턴직   
 3  서울 구로구 | 경력 3~7년 | 학력무관 | 정규직   
 4      서울 강남구 | 경력무관 | 석사↑ | 정규직   
 
                                              Col_url  
 0  https://www.saramin.co.kr/zf_user/jobs/relay/v...  
 1  https://www.saramin.co.kr/zf_user/jobs/relay/v...  
 2  https://www.saramin.co.kr/zf_user/jobs/relay/v...  
 3  https://www.saramin.co.kr/zf_user/jobs/relay/v...  
 4  https://www.saramin.co.kr/zf_user/jobs/relay/v...  )